In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt


In [ ]:
df = pd.read_csv("../data/time_series.csv")
values = df.value.values.reshape(-1,1)
scaler = MinMaxScaler()
values = scaler.fit_transform(values)


In [ ]:
class SeqDataset(torch.utils.data.Dataset):
    def __init__(self, data, seq_len=60):
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - seq_len

    def __getitem__(self, idx):
        seq = self.data[idx:idx+self.seq_len]
        target = self.data[idx+self.seq_len]
        return torch.FloatTensor(seq), torch.FloatTensor(target)


In [ ]:
class TransformerModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Linear(1, 64)
        layer = nn.TransformerEncoderLayer(d_model=64, nhead=4, batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=2)
        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        x = self.embedding(x)
        out = self.transformer(x)
        out = self.fc(out[:, -1])
        return out


In [ ]:
dataset = SeqDataset(values, seq_len=60)
loader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True)

model = TransformerModel()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

for epoch in range(40):
    for seq, target in loader:
        optimizer.zero_grad()
        pred = model(seq)
        loss = criterion(pred, target)
        loss.backward()
        optimizer.step()
    print(epoch, loss.item())


In [ ]:
test_input = values[-60:].reshape(1,60,1)
preds = []

for _ in range(20):
    with torch.no_grad():
        out = model(torch.FloatTensor(test_input))
        preds.append(out.item())
        test_input = np.append(test_input[:,1:,:], [[out]], axis=1)

preds = scaler.inverse_transform(np.array(preds).reshape(-1,1))
